# 부록 B. GitHub 저장소와 Colab 연동 · Appendix B — A GitHub Repository Wired to Colab

전산실험 2 · AI Agent 만들기 · **2주차** · 예상 소요 35분
Computational Experiments 2 · Building AI Agents · **Week 2** · about 35 minutes

## 목표 · Objectives
- 자신의 GitHub 저장소를 만든다
- Personal Access Token(PAT)을 발급해 안전하게 보관한다
- `clone` / `push` / `pull` 을 수행한다
- 충돌이 무엇인지 직접 만들어 본다
- **Colab 링크의 구조**를 이해하고 자기 노트북 링크를 만든다

> **EN**
> - Create your own GitHub repository
> - Issue a Personal Access Token (PAT) and store it safely
> - Perform `clone` / `push` / `pull`
> - Create a merge conflict on purpose, to see what one is
> - Understand **the structure of a Colab link** and build one for your own notebook

> **이 노트북은 2주차 부록입니다.** 본 진도(예제 01~03: 토큰·컨텍스트·비용)와
> 별개로, 앞으로 12주 동안 쓸 작업 환경을 갖추는 인프라 실습입니다.
> 오늘 못 끝내도 됩니다. 다만 **4주차 전에는 반드시 마쳐 두십시오.**
>
> **EN — this notebook is a week-2 appendix.** Separate from the main track
> (Examples 01–03: tokens, context, cost), it sets up the working environment you
> will use for the next twelve weeks. You need not finish it today, but
> **you must finish it before week 4.**

---
## 0. 준비 (수업 중 함께 진행합니다) · Setup — we do this together in class

### (1) 저장소 만들기 · Create the repository
1. https://github.com 로그인 → 우상단 **+** → **New repository**
2. 이름: `stat-agent-2026` (자유롭게 정해도 됩니다)
3. **Public** 선택 · **Add a README file** 체크
4. **Create repository**

> 공개(Public)를 권합니다. 학기말에 이 저장소가 포트폴리오가 됩니다.
> 대신 **비밀정보를 절대 올리면 안 됩니다.**

> **EN** — Sign in to GitHub, click **+ → New repository**, name it
> `stat-agent-2026` (any name is fine), choose **Public**, tick **Add a README
> file**, and create it. Public is recommended: by the end of term this
> repository is your portfolio. In exchange, **never commit a secret to it.**

### (2) PAT 발급 · Issue a PAT
1. 우상단 프로필 → **Settings**
2. 맨 아래 **Developer settings**
3. **Personal access tokens → Fine-grained tokens → Generate new token**
4. 설정
   - Expiration: 학기 말 이후 날짜
   - Repository access: **Only select repositories** → 방금 만든 저장소만
   - Permissions → Repository permissions → **Contents: Read and write**
5. **Generate token** → 나온 문자열을 **즉시 복사**

> 이 화면을 벗어나면 토큰을 다시 볼 수 없습니다.

> **EN** — Profile → **Settings** → **Developer settings** → **Personal access
> tokens → Fine-grained tokens → Generate new token**. Set the expiry past the end
> of term, restrict repository access to **only** the repository you just made,
> and grant **Contents: Read and write**. Generate it and **copy the string
> immediately** — once you leave that page you can never see it again.

### (3) Colab Secrets 에 저장 · Store it in Colab Secrets
왼쪽 **열쇠 아이콘(🔑)** → 새 secret
- `GITHUB_TOKEN` = 방금 복사한 토큰
- `GITHUB_USER` = 본인 GitHub 아이디

둘 다 이 노트북의 **액세스 토글을 켜세요.**

> **EN** — Key icon (🔑) on the left → new secret: `GITHUB_TOKEN` for the token
> you just copied and `GITHUB_USER` for your GitHub handle. **Turn on notebook
> access** for both.

In [ ]:
# 설정값 읽어오기 / Load the configuration values
import os

def get_secret(name):
    # EN — Colab Secrets -> environment variable -> prompt.
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v.strip()
    except Exception:
        pass
    if os.environ.get(name):
        return os.environ[name].strip()
    import getpass
    v = getpass.getpass(f"{name} 를 입력하세요: ").strip()
    os.environ[name] = v
    return v

GITHUB_USER  = get_secret("GITHUB_USER")
GITHUB_TOKEN = get_secret("GITHUB_TOKEN")
REPO_NAME    = "stat-agent-2026"      # <-- 자기 저장소 이름으로 바꾸세요 / use your own repository name

print("사용자:", GITHUB_USER)
print("저장소:", REPO_NAME)
print("토큰 길이:", len(GITHUB_TOKEN), "글자")   # 토큰 자체는 출력하지 않습니다 / never print the token itself

In [ ]:
# Git 사용자 정보 설정 / Configure the Git identity
!git config --global user.name  "{GITHUB_USER}"
!git config --global user.email "{GITHUB_USER}@users.noreply.github.com"
!git config --global init.defaultBranch main
print("설정 완료")

> `@users.noreply.github.com` 는 GitHub가 제공하는 **이메일 비공개용 주소**입니다.
> 실제 이메일을 커밋 이력에 남기고 싶지 않을 때 씁니다.

> **EN** — `@users.noreply.github.com` is GitHub's **private email address**, for
> when you would rather not leave your real address in the commit history.

---
## 1. 저장소 내려받기 (clone) · Cloning the repository

토큰을 주소에 끼워 넣으면 인증이 자동으로 됩니다.

> 아래 셀은 토큰이 들어간 주소를 **화면에 출력하지 않도록** 조심해서 작성했습니다.
> 실습 중에도 주소를 `print` 하지 마세요.

> **EN** — Embedding the token in the URL authenticates automatically. The cell
> below is written carefully so that the URL containing the token is **never
> printed**. Do not `print` that URL during the lab either.

In [ ]:
%cd /content
!rm -rf /content/{REPO_NAME}

# 토큰이 포함된 주소 (출력하지 않습니다)
# EN — The URL with the token embedded. Never print it.
clone_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git"

import subprocess
result = subprocess.run(["git", "clone", clone_url],
                        capture_output=True, text=True)
# 혹시 오류 메시지에 토큰이 섞여 있을 수 있으므로 가려서 출력
# EN — An error message may contain the token, so mask it before printing.
print(result.stderr.replace(GITHUB_TOKEN, "***TOKEN***"))
print("종료코드:", result.returncode)

In [ ]:
%cd /content/{REPO_NAME}
!ls -a
!git log --oneline

---
## 2. 파일 추가하고 올리기 (push) · Adding files and pushing

이 저장소가 앞으로 12주 동안 여러분의 작업 공간입니다.
먼저 폴더 구조부터 잡아 둡니다.

> **EN** — This repository is your workspace for the next twelve weeks. Start by
> laying out the folder structure.

In [ ]:
import os

for folder in ["prompts", "notebooks", "src", "data"]:
    os.makedirs(folder, exist_ok=True)
    # 빈 폴더는 git이 추적하지 않으므로 표시용 파일을 넣는다
    # EN — Git does not track empty folders, so drop in a placeholder file.
    with open(f"{folder}/.gitkeep", "w") as f:
        f.write("")

# .gitignore 를 먼저 만든다 -- 사고를 예방하는 순서
# EN — Write .gitignore FIRST; that ordering is what prevents accidents.
with open(".gitignore", "w", encoding="utf-8") as f:
    f.write("\n".join([
        "# 비밀정보 - 절대 커밋 금지",
        ".env", "*.key", "secrets.json",
        "",
        "# 파이썬",
        "__pycache__/", "*.pyc", ".ipynb_checkpoints/",
        "",
        "# 데이터",
        "data/*.csv",
        "",
    ]))

!ls -a
!cat .gitignore

In [ ]:
# 1주차 프롬프트를 저장소에 남긴다 / Commit the week-1 prompt to the repository
with open("prompts/guess_number.txt", "w", encoding="utf-8") as f:
    f.write(
        "1부터 100 사이의 정수를 하나만 고르세요.\n"
        "이분탐색 전략을 사용하세요: 가능한 범위의 중간값을 시도하세요.\n"
        "반드시 숫자만 답하고 다른 말은 하지 마세요.\n"
    )

!git add .
!git status --short

In [ ]:
!git commit -m "폴더 구조와 .gitignore 추가, 1주차 프롬프트 저장"

result = subprocess.run(["git", "push"], capture_output=True, text=True)
print(result.stdout.replace(GITHUB_TOKEN, "***TOKEN***"))
print(result.stderr.replace(GITHUB_TOKEN, "***TOKEN***"))
print("종료코드:", result.returncode)

브라우저에서 `https://github.com/<본인아이디>/stat-agent-2026` 를 열어 확인하세요.

> **EN** — Open `https://github.com/<your-handle>/stat-agent-2026` in a browser to
> confirm.

### push 가 안 될 때 · When push fails

| 메시지 | 원인 | 해결 |
|---|---|---|
| `403 Forbidden` | 토큰 권한 부족 | PAT의 **Contents: Read and write** 확인 |
| `Authentication failed` | 토큰 오타·만료 | 재발급 후 Secrets 갱신 |
| `Updates were rejected` | 원격이 앞서 있음 | `!git pull --rebase` 후 다시 push |
| `src refspec main does not match` | 브랜치 이름 불일치 | `!git branch -M main` |

**EN**

| Message | Cause | Fix |
|---|---|---|
| `403 Forbidden` | Token lacks permission | Check **Contents: Read and write** on the PAT |
| `Authentication failed` | Typo or expired token | Reissue it and update the secret |
| `Updates were rejected` | The remote is ahead | `!git pull --rebase`, then push again |
| `src refspec main does not match` | Branch-name mismatch | `!git branch -M main` |

---
## 3. 충돌 만들어 보기 · Creating a conflict on purpose

혼자 써도 충돌은 생깁니다. **Colab 런타임 두 개**에서 같은 파일을 고치거나,
GitHub 웹에서 고친 뒤 로컬에서도 고치면 그렇습니다.

일부러 한 번 만들어 봅시다.

> **EN** — Conflicts happen even when you work alone: edit the same file from
> **two Colab runtimes**, or edit it on the GitHub website and then locally too.
> Let's produce one deliberately.

**지금 브라우저에서 하세요:**

1. GitHub 저장소 → `prompts/guess_number.txt` 클릭
2. 연필 아이콘(Edit) 클릭
3. 맨 아래에 `# 웹에서 수정함` 한 줄 추가
4. **Commit changes**

끝났으면 아래 셀을 실행하세요.

> **EN — do this in the browser now:** open your repository, click
> `prompts/guess_number.txt`, click the pencil (Edit) icon, add the single line
> `# 웹에서 수정함` at the bottom, and **Commit changes**. Then run the cell below.

In [ ]:
# 로컬(Colab)에서도 같은 파일을 고친다 / Edit the same file locally, in Colab
with open("prompts/guess_number.txt", "a", encoding="utf-8") as f:
    f.write("# Colab에서 수정함\n")

!git add .
!git commit -m "Colab에서 주석 추가"

# 이제 push 하면? -- 거부됩니다 / Now try to push — it gets rejected
result = subprocess.run(["git", "push"], capture_output=True, text=True)
print(result.stderr.replace(GITHUB_TOKEN, "***TOKEN***"))

In [ ]:
# 해결: 원격 내용을 먼저 가져온다 / The fix: fetch the remote content first
result = subprocess.run(["git", "pull", "--rebase"], capture_output=True, text=True)
print(result.stdout.replace(GITHUB_TOKEN, "***TOKEN***"))
print(result.stderr.replace(GITHUB_TOKEN, "***TOKEN***"))

!git status

In [ ]:
# 충돌이 났다면 파일 안이 이렇게 되어 있습니다
# EN — If there was a conflict, this is what the file now looks like
!cat prompts/guess_number.txt

충돌 표시의 뜻:

```
<<<<<<< HEAD
원격(GitHub)에 있던 내용        what was on the remote (GitHub)
=======
내(Colab)가 쓴 내용             what I wrote (in Colab)
>>>>>>> 커밋해시                 commit hash
```

**해결 방법**: 파일을 열어 `<<<<<<<`, `=======`, `>>>>>>>` 세 줄을 지우고
남길 내용만 남긴 뒤 `git add` → `git rebase --continue` 합니다.

> **EN — how to resolve it**: open the file, delete the three marker lines
> `<<<<<<<`, `=======` and `>>>>>>>`, keep only the content you want, then
> `git add` followed by `git rebase --continue`.

In [ ]:
# 두 수정을 모두 살리는 방향으로 해결
# EN — Resolve it by keeping both edits
해결본 = (
    "1부터 100 사이의 정수를 하나만 고르세요.\n"
    "이분탐색 전략을 사용하세요: 가능한 범위의 중간값을 시도하세요.\n"
    "반드시 숫자만 답하고 다른 말은 하지 마세요.\n"
    "# 웹에서 수정함\n"
    "# Colab에서 수정함\n"
)
with open("prompts/guess_number.txt", "w", encoding="utf-8") as f:
    f.write(해결본)

!git add prompts/guess_number.txt
!git rebase --continue || git commit -m "충돌 해결"

result = subprocess.run(["git", "push"], capture_output=True, text=True)
print(result.stderr.replace(GITHUB_TOKEN, "***TOKEN***"))
print("종료코드:", result.returncode)

> `git rebase --continue` 가 편집기를 띄우려 해서 멈추면
> `!git -c core.editor=true rebase --continue` 로 실행하세요.
> Colab에서는 대화형 편집기를 쓸 수 없습니다.

> **EN** — If `git rebase --continue` stalls trying to open an editor, run
> `!git -c core.editor=true rebase --continue` instead. Colab cannot run an
> interactive editor.

---
## 4. Colab 링크의 구조 · The anatomy of a Colab link

여러분이 지금 보고 있는 이 노트북의 주소를 뜯어 봅시다.

```
https://colab.research.google.com/github/zoomer1975-boop/ai_agent/blob/main/week02/05_appendix_github_colab.ipynb
                                   └계정┘ └저장소┘      └브랜치┘└─── 파일 경로 ───┘
                                   account  repository   branch        file path
```

규칙은 간단합니다.

```
https://colab.research.google.com/github/<계정>/<저장소>/blob/<브랜치>/<경로>.ipynb
https://colab.research.google.com/github/<account>/<repo>/blob/<branch>/<path>.ipynb
```

**GitHub에 있는 공개 노트북이면 무엇이든** 이렇게 열 수 있습니다.
여러분의 저장소도 마찬가지입니다.

> **EN** — The rule is simple, and **any public notebook on GitHub** opens this
> way — including the ones in your own repository.

In [ ]:
# 내 저장소의 노트북 Colab 링크 만들기
# EN — Build a Colab link for a notebook in your own repository
def colab_link(user, repo, path, branch="main"):
    return f"https://colab.research.google.com/github/{user}/{repo}/blob/{branch}/{path}"

# 이 노트북을 내 저장소에 사본으로 저장했다고 가정
# EN — Assuming you saved a copy of this notebook into your repository
예시 = colab_link(GITHUB_USER, REPO_NAME, "notebooks/week02_practice.ipynb")
print(예시)
print()
print("아직 그 파일이 없으므로 404가 뜹니다.")
print("Colab 메뉴 [파일 > GitHub에 사본 저장] 으로 올린 뒤 다시 열어 보세요.")

### 내 노트북을 GitHub에 저장하는 법 · Saving your notebook to GitHub

Colab 메뉴에서:
**파일 → GitHub에 사본 저장** → 저장소·경로·커밋 메시지 지정

`git add`/`commit`/`push` 를 대신 해 주는 기능입니다.
편하지만 **무엇이 일어나는지는 알고 쓰십시오.** 그래서 오늘 손으로 먼저 해 본 것입니다.

> **EN** — In the Colab menu: **File → Save a copy in GitHub**, then choose the
> repository, path and commit message. It performs `git add`/`commit`/`push` for
> you. Convenient — but **know what it is doing before you rely on it.** That is
> why we did it by hand first today.

---
## 연습문제 · Exercises

**문제 1.** Colab 메뉴의 **GitHub에 사본 저장**으로 이 노트북을 자신의 저장소
`notebooks/` 폴더에 저장하고, 만들어진 Colab 링크로 다시 열어 보시오.

**문제 2.** GitHub 저장소의 `README.md` 를 수정해서
이 수업에서 무엇을 만들 것인지 3줄로 적고 push 하시오.

**문제 3.** `git pull` 과 `git pull --rebase` 는 이력 모양이 어떻게 다른가?
`git log --graph --oneline --all` 로 확인하시오.

**문제 4.** 저장소를 **Private** 으로 바꾸면 Colab 링크는 어떻게 되는가?
직접 바꿔 보고 확인한 뒤 다시 Public 으로 되돌리시오.

> **EN — 1.** Use **File → Save a copy in GitHub** to store this notebook in the
> `notebooks/` folder of your repository, then reopen it through the resulting
> Colab link.
>
> **EN — 2.** Edit your repository's `README.md` to describe in three lines what
> you will build in this course, and push it.
>
> **EN — 3.** How does the history differ between `git pull` and
> `git pull --rebase`? Check with `git log --graph --oneline --all`.
>
> **EN — 4.** What happens to the Colab link if you make the repository
> **Private**? Try it, confirm, then set it back to Public.

In [ ]:
# 여기에 직접 작성해 보세요 / Write your own code here

<details>
<summary><b>해설 보기 · Show the solutions</b></summary>

**문제 3.**
`git pull` 은 병합(merge) 커밋을 하나 더 만듭니다. 이력이 갈라졌다 합쳐지는
그래프가 됩니다. `--rebase` 는 내 커밋을 원격 커밋 **뒤로 옮겨 붙여** 이력을
한 줄로 만듭니다. 혼자 쓰는 저장소에서는 `--rebase` 쪽이 보기 깔끔합니다.

**문제 4.**
Private 저장소의 노트북은 `colab.research.google.com/github/...` 링크로 열리지
않습니다(404). Colab이 GitHub 계정 인증을 요구합니다.
→ **수업 자료 저장소는 Public 이어야 링크 배포가 됩니다.**
비밀정보를 커밋하면 안 되는 이유가 하나 더 늘었습니다.

---

**EN — 3.** `git pull` creates an extra merge commit, giving a graph that forks
and rejoins. `--rebase` **replays your commits on top of** the remote ones,
leaving a single straight line. In a repository you work in alone, `--rebase`
reads more cleanly.

**EN — 4.** A notebook in a private repository does not open through a
`colab.research.google.com/github/...` link (404); Colab demands GitHub account
authentication. → **A course-materials repository has to be public for link
sharing to work** — one more reason never to commit a secret.

</details>

---
## 정리 · Summary

- 인증은 비밀번호가 아니라 **PAT**. 권한은 최소로, 만료일은 짧게
- `clone` → 작업 → `add` → `commit` → `push`
- 충돌은 정상이다. `<<<<<<<` 표시를 지우고 남길 내용만 남긴다
- Colab 링크 = `colab.research.google.com/github/` + GitHub 경로
- **토큰을 화면에 출력하지 않는다** (이 노트북의 `replace(TOKEN, "***")` 참고)

> **EN**
> - Authentication is by **PAT**, not a password — least privilege, short expiry
> - `clone` → work → `add` → `commit` → `push`
> - Conflicts are normal: delete the `<<<<<<<` markers and keep what you want
> - A Colab link = `colab.research.google.com/github/` + the GitHub path
> - **Never print the token** — see this notebook's `replace(TOKEN, "***")`

**다음 노트북** → [`03_kmu_llm_package.ipynb`](https://colab.research.google.com/github/zoomer1975-boop/ai_agent/blob/main/week02/03_kmu_llm_package.ipynb)
오늘의 핵심 실습입니다.